In [87]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
import numpy as np
import h5py
import re

# root = "/home/lwu/work/NarrativeGenomics/region_narration/code/pipeline/ARCHS4_LW/refining/data_slash_results/"
# filename = root + 'archs4_gene_human_v2.1_normalized_LW20230126.h5'

filename = "/home/lwu/ARCHS4/Tests/top_10_genesets_per_category/test_cases/test_h5df_data.hdf5"

with h5py.File(filename, 'r') as f:
    qn_matrix = f['data/qn_matrix'][:]
    gene_names = f['meta/genes/all_gene_names'][:]
    category_names = f['meta/biosamples/category'][:]

category_names = [x.decode('UTF-8') for x in category_names]
gene_names = [x.decode('UTF-8') for x in gene_names]

# Variables 
category_dict  = {
    "Cardiovascular System" : ["Heart", "Atrium", "Cardiac Muscle Fiber", 
                               "Pericardium", "Valve", "Ventricle",
                               "HUVEC"], 
    
    "Connective Tissue" : ["Adipose", "Adipocyte", "Myofibroblast", "Osteoblast", "Stromal Cell", 
                           "MG63", "K562", "Bone Marrow", "Chondrocyte", "HT1080", "NHDF"],

    "Digestive System": ["HCT116", "HT29", "RKO", "Colon", "Colonic Muscosa", "Intestinal Epithelial Cell", 
                         "Small Intestine", "Liver", "Hepatic Stellate Cell", "Hepatocyte", "Primary Liver Cell",
                         "HEPG2", "Alpha Cell", "Beta Cell", "Pancreatic Islet", 
                         "GSE81547", "PANC1", "Gastric Epithelial Cell", "Esophagus"],
    
    "Immune System" : ["Granulocyte", "Neutrophil", "Blymphocyte", "Plasma Cell", "Plasmacytoid Dendritic Cell", 
                       "Tlymphocyte", "GSM2679484", "JURKAT", "MT4", "NALM6", "REH", "RPMI8226", "Alveolar", 
                       "Dendritic", "Kupffer", "Macrophage", "Macroglia",  "THP1", "U937", "JEG3", "Thymus", 
                       "Thymocyte", "Spleen"],
    
    "Integumentary System" : ["Skin", "Basal Cell", "Fibroblast", "Hair Follicle", "Keratinocyte", "Melanocyte",
                              "A431", "HACAT", "HNSCC"], 
    
    "Muscular System" : ["Skeletal Muscle", "Myoblast", "Airway Smooth Muscle", "Respiratory Smooth Muscle", 
                         "Vascular Smooth Muscle", "LHCNM2"], 
    
    "Nervous System" : ["Brain", "DAOY", "IMR32", "SKNSH", "Astrocyte", "Cerebellum", "Cerebral Cortex", 
                        "Hypothalamus", "Medula", "Midbrain", "Neuron", "Oligodendrocyte", "Pons", "Spinal Cord", 
                        "Thalamus", "Retina", "Motor Neuron", "Sensory Neuron"], 
    
    "Respiratory System" : ["Lung", "Lung Epithelial", "A549", "H1299", "IMR90", "MRC5", "NCIH460", "NHBE", 
                            "Trachea"], 
    
    "Urogenital System" : ["Breast", "Mammary Gland", "MCF7", "MCF10", "MDAMB231", "MDAMB468", "SKBR3", "T47D", 
                           "HELA", "Kidney", "Podocyte", "Renal Cortex", "293F", "FLPIN TREX 293", "HEK293", "C42", 
                           "DU145", "LNCAP", "PC3", "Ovary", "Granulosa", "Oocyte", "ES2", "SKOV3", "Testis"]
}

# folder = "/home/lwu/ARCHS4/Tests/Check If Biosample Categorys Align With Genesets/"
# filename = folder + "h.c1.c2.all.v7.5.1.symbols.gmt"

filename = "/home/lwu/ARCHS4/Tests/top_10_genesets_per_category/test_cases/test_geneset.gmt"

In [88]:
for category, value_list in category_dict.items():
    for i in range(len(value_list)):
        value_list[i] = value_list[i].lower()

print(category_dict)


{'Cardiovascular System': ['heart', 'atrium', 'cardiac muscle fiber', 'pericardium', 'valve', 'ventricle', 'huvec'], 'Connective Tissue': ['adipose', 'adipocyte', 'myofibroblast', 'osteoblast', 'stromal cell', 'mg63', 'k562', 'bone marrow', 'chondrocyte', 'ht1080', 'nhdf'], 'Digestive System': ['hct116', 'ht29', 'rko', 'colon', 'colonic muscosa', 'intestinal epithelial cell', 'small intestine', 'liver', 'hepatic stellate cell', 'hepatocyte', 'primary liver cell', 'hepg2', 'alpha cell', 'beta cell', 'pancreatic islet', 'gse81547', 'panc1', 'gastric epithelial cell', 'esophagus'], 'Immune System': ['granulocyte', 'neutrophil', 'blymphocyte', 'plasma cell', 'plasmacytoid dendritic cell', 'tlymphocyte', 'gsm2679484', 'jurkat', 'mt4', 'nalm6', 'reh', 'rpmi8226', 'alveolar', 'dendritic', 'kupffer', 'macrophage', 'macroglia', 'thp1', 'u937', 'jeg3', 'thymus', 'thymocyte', 'spleen'], 'Integumentary System': ['skin', 'basal cell', 'fibroblast', 'hair follicle', 'keratinocyte', 'melanocyte', 'a4

In [89]:
qn_matrix = [[ 0,  1,  2,  3,  4],
             [ 5,  6,  7,  8,  9],
             [10, 11, 12, 13, 14],
             [15, 16, 17, 18, 19],
             [20, 21, 22, 23, 24]]

gene_names = ["gene1", "gene2", "gene3", "gene4", "gene5"]

category_names =["heart", "atrium", "heart",  "adipose", "adipose2"]

# category_dict = {"Cardiovascular System": ["Heart", "Atrium"],
#                  "Connective Tissue": ["Adipose", "Adipocyte"]}

# Initialize the dictionaries to store the averages
category_averages = {}
tissue_averages = {}
for category in category_dict:
    category_averages[category] = {}
    for tissue in category_dict[category]:
        category_averages[category][tissue] = []
    tissue_averages[category] = []

# Calculate the averages
for i, tissue in enumerate(category_names):
    category = None
    for c in category_dict:
        if tissue in category_dict[c]:
            category = c
            break
    if category is not None:
        for j, gene in enumerate(gene_names):
            value = qn_matrix[i][j]
            category_averages[category][tissue].append(value)
            tissue_averages[category].append(value)
for category in category_averages:
    for tissue in category_averages[category]:
        n_samples = len(category_dict[category])
        category_averages[category][tissue] = sum(category_averages[category][tissue]) / n_samples
    tissue_averages[category] = sum(tissue_averages[category]) / len(category_names)

# Print the resulting dictionaries
print("Category averages:")
print(category_averages)
print("\nTissue averages:")
print(tissue_averages)


Category averages:
{'Cardiovascular System': {'heart': 10.0, 'atrium': 5.0, 'cardiac muscle fiber': 0.0, 'pericardium': 0.0, 'valve': 0.0, 'ventricle': 0.0, 'huvec': 0.0}, 'Connective Tissue': {'adipose': 7.7272727272727275, 'adipocyte': 0.0, 'myofibroblast': 0.0, 'osteoblast': 0.0, 'stromal cell': 0.0, 'mg63': 0.0, 'k562': 0.0, 'bone marrow': 0.0, 'chondrocyte': 0.0, 'ht1080': 0.0, 'nhdf': 0.0}, 'Digestive System': {'hct116': 0.0, 'ht29': 0.0, 'rko': 0.0, 'colon': 0.0, 'colonic muscosa': 0.0, 'intestinal epithelial cell': 0.0, 'small intestine': 0.0, 'liver': 0.0, 'hepatic stellate cell': 0.0, 'hepatocyte': 0.0, 'primary liver cell': 0.0, 'hepg2': 0.0, 'alpha cell': 0.0, 'beta cell': 0.0, 'pancreatic islet': 0.0, 'gse81547': 0.0, 'panc1': 0.0, 'gastric epithelial cell': 0.0, 'esophagus': 0.0}, 'Immune System': {'granulocyte': 0.0, 'neutrophil': 0.0, 'blymphocyte': 0.0, 'plasma cell': 0.0, 'plasmacytoid dendritic cell': 0.0, 'tlymphocyte': 0.0, 'gsm2679484': 0.0, 'jurkat': 0.0, 'mt4': 

In [90]:
tissue_averages

{'Cardiovascular System': 21.0,
 'Connective Tissue': 17.0,
 'Digestive System': 0.0,
 'Immune System': 0.0,
 'Integumentary System': 0.0,
 'Muscular System': 0.0,
 'Nervous System': 0.0,
 'Respiratory System': 0.0,
 'Urogenital System': 0.0}

In [91]:
category_averages = {}
tissue_averages = {}
for category in category_dict:
    category_averages[category] = {}
    for tissue in category_dict[category]:
        category_averages[category][tissue] = []
    tissue_averages[category] = []


# Iterate through each row in the matrix
for row in qn_matrix:
    # Initialize a dictionary to hold the sample values for each category
    category_values = {category_name: [] for key, value in category_dict}
    
    print(category_values)
    # Iterate through each sample in the row
    for sample_idx, sample_value in enumerate(row):
        sample_name = gene_names[sample_idx]
        
        # Check which categories the sample belongs to
        for category_name, category_values_list in category_dict.items():
            if set(category_values_list).issubset(set(sample_name.split('_'))):
                category_values[category_name].append(sample_value)
    
    # Compute the average for each category
    category_averages = [sum(values) / len(values) if len(values) > 0 else 0 for values in category_values.values()]
    
    # Append the category averages for the current row to the list of all row averages
    averages.append(category_averages)

# Print the list of row averages
print(averages)


ValueError: too many values to unpack (expected 2)

In [92]:
averages

[[0, 0], [0, 0], [0, 0], [0, 0], [0, 0]]

In [93]:
gene_system_idx = {}

for element in category_names:
    print(element)
    for category, keywords in category_dict.items():
        if any(keyword in element for keyword in keywords):
            if category not in gene_system_idx:
                gene_system_idx[category] = []
            indexes = [i for i, sublist in enumerate(category_names) if element in sublist]
            gene_system_idx[category].extend(indexes)

print(gene_system_idx)

gene_cat_avgs = []
for gene_row in qn_matrix:
    cat_avg = []

    for system, indexes in gene_system_idx.items():
        values = [gene_row[index] for index in indexes]
        mean = sum(values)/len(values)
        cat_avg.append(mean)
    gene_cat_avgs.append(cat_avg)
print(gene_cat_avgs)


heart
atrium
heart
adipose
adipose2
{'Cardiovascular System': [0, 2, 1, 0, 2], 'Connective Tissue': [3, 4, 4]}
[[1.0, 3.6666666666666665], [6.0, 8.666666666666666], [11.0, 13.666666666666666], [16.0, 18.666666666666668], [21.0, 23.666666666666668]]


In [94]:
gene_system_idx = {}

for element in category_names:
    for category, keywords in category_dict.items():
        if any(keyword in element for keyword in keywords):
            if category not in gene_system_idx:
                gene_system_idx[category] = []
            indexes = [i for i, sublist in enumerate(category_names) if element in sublist]
            print(indexes)
            gene_system_idx[category].extend(indexes)

[0, 2]
[1]
[0, 2]
[3, 4]
[4]


In [95]:
gene_system_idx

{'Cardiovascular System': [0, 2, 1, 0, 2], 'Connective Tissue': [3, 4, 4]}

In [96]:
category_names

['heart', 'atrium', 'heart', 'adipose', 'adipose2']

In [97]:
qn_matrix = [[ 0,  1,  2,  3,  4],
             [ 5,  6,  7,  8,  9],
             [10, 11, 12, 13, 14],
             [15, 16, 17, 18, 19],
             [20, 21, 22, 23, 24]]

gene_names = ["gene1", "gene2", "gene3", "gene4", "gene5"]

category_names =["heart", "atrium", "heart",  "adipose", "adipose2"]

In [98]:
import numpy as np

qn_matrix = [[ 0,  1,  2,  3,  4],
             [ 5,  6,  7,  8,  9],
             [10, 11, 12, 13, 14],
             [15, 16, 17, 18, 19],
             [20, 21, 22, 23, 24]]
gene_names = ["gene1", "gene2", "gene3", "gene4", "gene5"]
category_names =["heart1", "atrium1", "heart2",  "adipose1", "adipose2"]
category_dict = {"Cardiovascular System": ["Heart", "Atrium"],
                 "Connective Tissue": ["Adipose", "Adipocyte"]}
category_list = ["Heart", "Atrium", "Adipose", "Adipocyte"]

# Convert category_dict value list elements to lowercase
for category, value_list in category_dict.items():
    for i in range(len(value_list)):
        value_list[i] = value_list[i].lower()
# print(category_dict)

gene_system_idx = {}

for element in category_names:
    for category, keywords in category_dict.items():
        if any(keyword in element for keyword in keywords):
            if category not in gene_system_idx:
                gene_system_idx[category] = []
            indexes = [i for i, sublist in enumerate(category_names) if element in sublist]
            gene_system_idx[category].extend(indexes)

print(gene_system_idx)

gene_cat_avgs = []
for gene_row in qn_matrix:
    cat_avg = []

    for system, indexes in gene_system_idx.items():
        values = [gene_row[index] for index in indexes]
        mean = sum(values)/len(values)
        cat_avg.append(mean)
    gene_cat_avgs.append(cat_avg)
print(gene_cat_avgs)



{'Cardiovascular System': [0, 1, 2], 'Connective Tissue': [3, 4]}
[[1.0, 3.5], [6.0, 8.5], [11.0, 13.5], [16.0, 18.5], [21.0, 23.5]]


In [99]:
indexes_2d_list

{'Cardiovascular System': [0, 1, 2], 'Connective Tissue': [3, 4]}

In [100]:
import numpy as np

qn_matrix = [[ 0,  1,  2,  3,  4],
             [ 5,  6,  7,  8,  9],
             [10, 11, 12, 13, 14],
             [15, 16, 17, 18, 19],
             [20, 21, 22, 23, 24]]
gene_names = ["gene1", "gene2", "gene3", "gene4", "gene5"]
category_names =["heart", "atrium1", "heart2",  "adipose1", "adipose2"]
category_dict = {"Cardiovascular System": ["Heart", "Atrium"],
                 "Connective Tissue": ["Adipose", "Adipocyte"]}
category_list = ["Heart", "Atrium", "Adipose", "Adipocyte"]

# Convert category_dict value list elements to lowercase
for category, value_list in category_dict.items():
    for i in range(len(value_list)):
        value_list[i] = value_list[i].lower()
# print(category_dict)

gene_system_idx = {}
for category, keywords in category_dict.items():
    category_indexes = []
    for i, sample_name in enumerate(category_names):
        for keyword in keywords:
            if keyword.lower() in sample_name.lower():
                category_indexes.append(i)
                break
    gene_system_idx[category] = category_indexes
    
gene_cat_avgs = []
for gene_row in qn_matrix:
    cat_avg = []

    for system, indexes in gene_system_idx.items():
        values = [gene_row[index] for index in indexes]
        mean = sum(values)/len(values)
        cat_avg.append(mean)
    gene_cat_avgs.append(cat_avg)
print(gene_cat_avgs)




[[1.0, 3.5], [6.0, 8.5], [11.0, 13.5], [16.0, 18.5], [21.0, 23.5]]
